<a href="https://colab.research.google.com/github/rohkdev/ClimateModelingResearch/blob/main/Task%202%3A%20Valid%20Stations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the candidate table you already uploaded
stations_df = pd.read_csv("Research_Dataset-candidate_station_table.csv")


In [ ]:
# Apply our Task #1 screening filters
filtered_stations = stations_df[
    (stations_df['number_of_usable_years'] >= 30) &
    (stations_df['percent_missing_summer_TMAX'] <= 10) &
    (stations_df['percent_missing_summer_TMIN'] <= 10) &
    (stations_df['elevation'] != -999.9)
]

# Create a set of valid station IDs for lightning-fast lookups
valid_station_ids = set(filtered_stations['station_id'])
print(f"Tracking {len(valid_station_ids)} high-quality stations.")

Tracking 325 high-quality stations.


In [ ]:
filtered_stations.head()

,station_id,latitude,longitude,elevation,state,name,available_TMAX_first_year,available_TMIN_first_year,available_TMAX_last_year,available_TMIN_last_year,first_year_daily,last_year_daily,number_of_years_with_any_summer_data,number_of_usable_years,mean_valid_summer_TMAX_days,mean_valid_summer_TMIN_days,percent_missing_summer_TMAX,percent_missing_summer_TMIN,passes_task1_filters,passes_task2_filters
12,USC00040192,33.8647,-117.8425,70.7,CA,ANAHEIM,1989.0,1989.0,2026.0,2026.0,1989.0,2025.0,37.0,33,115.756757,116.027027,5.117412,4.895879,True,True
14,USC00040212,38.5731,-122.4406,522.7,CA,ANGWIN PACIFIC UNION COLLEGE,1952.0,1952.0,2026.0,2026.0,1970.0,2026.0,57.0,51,115.842105,116.807018,5.047455,4.256543,True,True
16,USC00040232,37.9833,-121.7525,18.3,CA,ANTIOCH PUMPING PLT #3,1955.0,1955.0,2021.0,2021.0,1970.0,2021.0,51.0,39,111.784314,112.647059,8.373513,7.666345,True,True
20,USC00040343,36.4914,-118.8253,520.6,CA,ASH MTN,1927.0,1927.0,2026.0,2026.0,1970.0,2026.0,53.0,41,112.320755,111.962264,7.933808,8.227652,True,True
21,USC00040379,37.0919,-119.5128,637.0,CA,AUBERRY 2 NW,1915.0,1915.0,2019.0,2019.0,1970.0,2019.0,50.0,47,117.900000,119.620000,3.360656,1.950820,True,True


In [ ]:
import pandas as pd
import numpy as np

big_file_path = "Temperature Data.csv" # Update to your exact filename from the screenshot
output_file_path = "clean_daily_temperatures.csv"
metadata_file_path = "Research_Dataset-candidate_station_table.csv"

# 1. Load metadata and filter valid IDs
metadata_df = pd.read_csv(metadata_file_path, dtype={'station_id': str})
metadata_df['station_id'] = metadata_df['station_id'].str.strip()
valid_station_ids = set(metadata_df[metadata_df['passes_task1_filters'] == True]['station_id'])
spatial_lookup = metadata_df[['station_id', 'latitude', 'longitude', 'elevation']].drop_duplicates()

is_first_chunk = True
total_rows = 0

# 2. Process chunks
for chunk in pd.read_csv(big_file_path, chunksize=100000, low_memory=False, dtype={'station_id': str}):
    chunk['station_id'] = chunk['station_id'].str.strip()

    # Filter for your 325 valid stations
    chunk = chunk[chunk['station_id'].isin(valid_station_ids)].copy()
    if chunk.empty:
        continue

    # Merge spatial metadata (lat, lon, elevation)
    chunk = pd.merge(chunk, spatial_lookup, on='station_id', how='left')

    # Construct the date column from separate year, month, day columns
    try:
        chunk['day'] = pd.to_datetime(chunk[['year', 'month', 'day']], errors='coerce')
    except Exception:
        pass # fallback if columns differ

    # Restrict to study period (1970-present)
    chunk = chunk[chunk['day'].dt.year >= 1970]
    if chunk.empty:
        continue

    # Clean temperature values & compute TAVG_C midpoint
    chunk['TMAX'] = pd.to_numeric(chunk['TMAX'], errors='coerce')
    chunk['TMIN'] = pd.to_numeric(chunk['TMIN'], errors='coerce')
    chunk['TAVG_C'] = (chunk['TMAX'] + chunk['TMIN']) / 2

    # --- ADDED: Fahrenheit Conversions ---
    chunk['TMAX_F'] = (chunk['TMAX'] * 9/5) + 32
    chunk['TMIN_F'] = (chunk['TMIN'] * 9/5) + 32
    chunk['TAVG_F'] = (chunk['TAVG_C'] * 9/5) + 32

    # Save output
    if is_first_chunk:
        chunk.to_csv(output_file_path, mode='w', index=False)
        is_first_chunk = False
    else:
        chunk.to_csv(output_file_path, mode='a', header=False, index=False)

    total_rows += len(chunk)

print(f"Success! Processed {total_rows:,} rows across all valid stations.")

Success! Processed 69,161 rows across all valid stations.
